📘 Baseline Models – Collaboration Notebook (Block Structure)

This notebook contains the shared project structure for our team.
Each block represents a specific functional part of the baseline models (A, B, C, D).
Team members should write their names next to the blocks they take responsibility for.

Please do not implement code in this file yet — each person will complete their own block in their own branch.

🔷 Block 0 – Imports & Setup

Assigned to: UNASSIGNED
Description:
Load standard libraries, PyTorch modules, timm, and utility functions.
Define device (CUDA/CPU).

In [1]:
import sys
print(sys.executable)


c:\Users\Elahe\Python\miniAnacoda\python.exe


In [2]:
import torch
print(torch.__version__)
print("cuda:", torch.cuda.is_available())

2.6.0+cu124
cuda: True


In [18]:
#Block 0
import os
import warnings
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from pathlib import Path
import random
import numpy as np

FLOWER_CKPT = "../models/flower_resnet18_state.pth"
# Path to the flower-pretrained ResNet-18 checkpoint.
# This checkpoint is used as an intermediate "pertaining" step
# before adapting the model to the household 10-class task.

# Suppress cuDNN deterministic warning (performance trade-off for reproducibility)
# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# cuDNN settings for reproducibility (may slow down training)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Choose device: GPU if available, otherwise CPU.
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ CUDA GPU is available! Training on CUDA...")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU found. Training on CPU...")

print("Using device:", device)
warnings.filterwarnings("ignore", message=".*cuDNN.*")
warnings.filterwarnings("ignore", category=UserWarning, module="torch.backends.cudnn")

✅ CUDA GPU is available! Training on CUDA...
Using device: cuda


In [30]:
data_root = Path("data")
imagenet_root = data_root / "ImageNetSubset"  # Note: nested folder structure
train_dir = imagenet_root / "train"
val_dir   = imagenet_root / "val"


🔷 Block 1 – Dataset & DataLoader Setup

Assigned to: UNASSIGNED
Description:

Define data transforms

Load ImageNetSubset dataset

Create dataloaders for train/val

Print class names, dataset sizes

In [35]:
#Block 1
# Root directory of the household dataset (ImageNetSubset)
data_root = Path("data")
imagenet_root = data_root/ "ImageNetSubset"  # Note: nested folder structure
train_dir = imagenet_root / "train"
val_dir   = imagenet_root / "val"

input_size = 224

data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),
}

def has_images(directory: Path):
    """Check if directory has images (case-insensitive: jpg, jpeg, etc.)"""
    if not directory.exists():
        return False
    valid_extensions = {".jpg", ".jpeg",".JPG", ".JPEG"}
    for f in directory.rglob("*"):
        if f.suffix.lower() in valid_extensions:
            return True
    return False

# Create dataset objects for training and validation
if has_images(train_dir) and has_images(val_dir):
    print("✅ Using real ImageNetSubset dataset (ImageFolder)")

    image_datasets = {
        "train": datasets.ImageFolder(train_dir, data_transforms["train"]),
        "val": datasets.ImageFolder(val_dir, data_transforms["val"]),
    }
    num_classes = len(image_datasets["train"].classes)
    class_names = image_datasets["train"].classes

else:
    print("⚠️ No real dataset found. Using FakeData for pipeline validation.")

    num_classes = 10
    class_names = [f"class_{i}" for i in range(num_classes)]
    image_datasets = {
        "train": datasets.FakeData(
            size=64,
            image_size=(3, input_size, input_size),
            num_classes=num_classes,
            transform=data_transforms["train"],
        ),
        "val": datasets.FakeData(
            size=32,
            image_size=(3, input_size, input_size),
            num_classes=num_classes,
            transform=data_transforms["val"],
        ),
    }

# DataLoader
batch_size = 32

# ✅ For notebook + WSL stability, start with num_workers=0
# You can increase it later for real training.
num_workers = 0

dataloader = {
    "train": DataLoader(image_datasets["train"], batch_size=batch_size, shuffle=True,  num_workers=num_workers),
    "val":   DataLoader(image_datasets["val"],   batch_size=batch_size, shuffle=False, num_workers=num_workers),
}

print("Train samples:", len(image_datasets["train"]))
print("Val samples:", len(image_datasets["val"]))
print("Num classes:", num_classes)
print("Classes:", class_names)

✅ Using real ImageNetSubset dataset (ImageFolder)
Train samples: 13000
Val samples: 500
Num classes: 10
Classes: ['binder', 'coffee_mug', 'computer_keyboard', 'mouse', 'notebook', 'remote_control', 'soup_bowl', 'teapot', 'toilet_tissue', 'wooden_spoon']


🔷 Block 2 – Training Function (train_model)

Assigned to: UNASSIGNED
Description:
Provide a reusable training loop supporting:

train & eval mode

loss/acc tracking

saving best checkpoint

printing epoch results

This function will be shared by Models A, B, C, D.

In [34]:
#Block 2
def train_model(
    model,
    train_loader,
    val_loader,
    device,
    num_epochs,
    lr=1e-4,
    save_path="../models/best_model.pth",
):
    """
    Generic training loop for a classification model.

    Returns:
        model: best model (based on highest validation accuracy)
        best_acc: best validation accuracy
        history: dict with epoch-wise metrics:
            - "train_loss": list[float]
            - "val_loss":   list[float]
            - "train_acc":  list[float]
            - "val_acc":    list[float]
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    # Only parameters with requires_grad=True will be updated
    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        momentum=0.9,
    )

    dataset_sizes = {
        "train": len(train_loader.dataset),
        "val": len(val_loader.dataset),
    }

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # ---- TRAIN PHASE ----
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == labels).sum().item()

        train_epoch_loss = running_loss / dataset_sizes["train"]
        train_epoch_acc = running_corrects / dataset_sizes["train"]
        history["train_loss"].append(train_epoch_loss)
        history["train_acc"].append(train_epoch_acc)

        print(f"train  Loss: {train_epoch_loss:.4f}  Acc: {train_epoch_acc:.4f}")

        # ---- VAL PHASE ----
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()

        val_epoch_loss = running_loss / dataset_sizes["val"]
        val_epoch_acc = running_corrects / dataset_sizes["val"]
        history["val_loss"].append(val_epoch_loss)
        history["val_acc"].append(val_epoch_acc)

        print(f"val    Loss: {val_epoch_loss:.4f}  Acc: {val_epoch_acc:.4f}")

        # save best model by validation accuracy
        if val_epoch_acc > best_acc:
            best_acc = val_epoch_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
            print(f"  ↳ New best! Saved to {save_path}")

    print(f"\nBest val Acc: {best_acc:.4f}")
    model.load_state_dict(best_state)

    return model, best_acc, history

🔷 Block 3 – Flower Pretrained Backbone Explanation

Assigned to: UNASSIGNED
Description:
Document how flower_resnet18_state.pth was created using train_flower_pretrain.py, including:

Dataset (Oxford Flowers 102)

Training approach

Extracting backbone weights (removing fc layer)

This block does not need training code — only explanation.

In [22]:
#Block 3

🔷 Block 4 – Model A (ImageNet Pretrained, Backbone Frozen)

Assigned to: UNASSIGNED
Description:

Create ResNet-18 with ImageNet weights

Replace final fc layer

Freeze backbone

Train & report accuracy

Represents a linear probe baseline.

In [23]:
#Block 4
from torchvision.models import ResNet18_Weights
def create_model_A(num_classes):
    """
    Model A:
    - ResNet-18 with ImageNet-pretrained weights.
    - Replace final fc layer with num_classes outputs.
    - Freeze backbone → only train the new fc layer (linear probing).
    """
    # 1. Load ImageNet-pretrained ResNet-18
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)

    # 2. Replace the final classifier for our household 10-class task
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # 3. Freeze all layers except the final fc layer
    for name, param in model.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = False

    print("Model A: ImageNet-pretrained backbone frozen (linear probe).")
    return model.to(device)

🔷 Block 5 – Model B (ImageNet Pretrained, Full Fine-Tuning)

Assigned to: UNASSIGNED
Description:

Same architecture as Model A

But do not freeze backbone

Train and compare against Model A

Represents a fine-tuned baseline.

In [24]:
#Block 5
def create_model_B(num_classes):
    """
    Model B:
    - Same as Model A (ImageNet-pretrained ResNet-18),
      but the whole network is trainable (full fine-tuning).
    """
    # 1. Load ImageNet-pretrained ResNet-18
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)

    # 2. Replace the final classifier for our household 10-class task
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # 3. Make all parameters trainable
    for name, param in model.named_parameters():
        param.requires_grad = True

    print("Model B: ImageNet-pretrained backbone trainable (fine-tuning).")
    return model.to(device)

🔷 Block 6 – Model C (Flower Pretrained, Backbone Frozen)

Assigned to: UNASSIGNED
Description:

Load ResNet-18

Load flower-pretrained weights (exclude fc)

Replace fc with household classifier

Freeze backbone

Train & compare to Model A

Tests whether flower pretraining provides better initialization than ImageNet.

In [25]:
#Block 6
def create_model_C(num_classes):
    """
    Model C:
    - Base architecture: ResNet-18
    - Initialization: from a 'flower-pretrained' checkpoint
      (ResNet-18 fine-tuned on a flower dataset).
    - Replace the final fc layer to adapt the model to 'num_classes'.
    - Freeze the backbone → only train the new fc layer (linear probing).
    """
    # 1. Create a fresh ResNet-18 (no weights)
    model = models.resnet18(weights=None)

    # 2. Load flower-pretrained checkpoint
    raw_state = torch.load(FLOWER_CKPT, map_location="cpu")

    # 2.1 Handle potential 'module.' prefix (if saved with DataParallel)
    if any(k.startswith("module.") for k in raw_state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in raw_state.items()}
    else:
        state = raw_state

    # 3. Remove fc weights from checkpoint → only load backbone
    state_no_fc = {k: v for k, v in state.items() if not k.startswith("fc.")}
    missing, unexpected = model.load_state_dict(state_no_fc, strict=False)

    print("Loaded flower-pretrained backbone (without fc).")
    print("Missing keys:", missing)       # typically ['fc.weight', 'fc.bias']
    print("Unexpected keys:", unexpected) # usually []

    # 4. New classifier for household task
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # 5. Freeze backbone, only fc is trainable
    for name, param in model.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = False

    print("Model C: Flower-pretrained backbone frozen (linear probe).")
    return model.to(device)

🔷 Block 7 – Model D (Flower Pretrained, Full Fine-Tuning)

Assigned to: UNASSIGNED
Description:

Same as Model C

Backbone is trainable

Train & compare to Model B

In [26]:
#Block 7
def create_model_D(num_classes):
    """
    Model D:
    - Same initialization as Model C (flower-pretrained ResNet-18),
      but the whole network is trainable (full fine-tuning).
    """
    # 1. Create a fresh ResNet-18 (no weights)
    model = models.resnet18(weights=None)

    # 2. Load flower-pretrained checkpoint
    raw_state = torch.load(FLOWER_CKPT, map_location="cpu")

    # 2.1 Handle 'module.' prefix if needed
    if any(k.startswith("module.") for k in raw_state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in raw_state.items()}
    else:
        state = raw_state

    # 3. Remove fc weights from checkpoint → only load backbone
    state_no_fc = {k: v for k, v in state.items() if not k.startswith("fc.")}
    missing, unexpected = model.load_state_dict(state_no_fc, strict=False)

    print("Loaded flower-pretrained backbone (without fc).")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    # 4. New classifier for household task
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # 5. Full fine-tuning → all parameters trainable
    for name, param in model.named_parameters():
        param.requires_grad = True

    print("Model D: Flower-pretrained backbone trainable (fine-tuning).")
    return model.to(device)

🔷 Block 8 – Comparison, Plots & Final Discussion

Assigned to: UNASSIGNED
Description:
Summarize results across Models A–D:

Table of accuracies

Discussion of differences

Insights: transferability from ImageNet vs. Flowers

In [27]:
#Block 8
# ===== Shared loaders and meta info =====
train_loader = dataloader['train']
val_loader   = dataloader['val']

# ⚠️ num_classes is already defined in Block 1; reuse it here
print("Number of classes:", num_classes)
print("Dataset type:", type(image_datasets["train"]).__name__)  # FakeData or ImageFolder

# Dictionary to store best validation accuracies of all models
results = {}

import os
os.makedirs("../models", exist_ok=True)  # Ensure the save directory exists

# ===== Model A: ImageNet-pretrained + frozen backbone (linear probe) =====
print("\n========== Training Model A ==========")
model_A = create_model_A(num_classes)

model_A, best_acc_A, hist_A = train_model(
    model=model_A,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=1,        # Self-check: run only 1 epoch
    lr=1e-3,
    save_path="../models/best_model_A.pth"
)

results["Model A"] = best_acc_A



Number of classes: 10
Dataset type: FakeData

========== Training Model A ==========
Model A: ImageNet-pretrained backbone frozen (linear probe).

Epoch 1/1


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
#Block 8
# ===== Shared loaders and meta info =====
train_loader = dataloader['train']
val_loader   = dataloader['val']
num_classes  = len(image_datasets['train'].classes)

print("Number of classes:", num_classes)

# Dictionary to store best validation accuracies of all models
results = {}

# ===== Model A: ImageNet-pretrained + frozen backbone (linear probe) =====
print("\n========== Training Model A ==========")
model_A = create_model_A(num_classes)

model_A, best_acc_A, hist_A = train_model(
    model=model_A,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=10,
    lr=1e-3,
    save_path="../models/best_model_A.pth"
)
results["Model A"] = best_acc_A

# ===== Model B: ImageNet-pretrained + unfrozen backbone (fine tuning) =====
print("\n========== Training Model B ==========")
model_B = create_model_B(num_classes)
model_B, best_acc_B, hist_B = train_model(
    model=model_B,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=10,
    lr=1e-4,   # fine-tuning usually use a smaller lr
    save_path="../models/best_model_B.pth"
)
results["Model B"] = best_acc_B

# ===== Model C: flower-pretrained + frozen backbone (linear probe) =====
print("\n========== Training Model C ==========")
model_C = create_model_C(num_classes)

model_C, best_acc_C, hist_C = train_model(
    model=model_C,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=10,
    lr=1e-3,
    save_path="../models/best_model_C.pth"
)
results["Model C"] = best_acc_C

# ===== Model D: flower-pretrained + unfrozen backbone (fine tuning) =====
print("\n========== Training Model D ==========")
model_D = create_model_D(num_classes)
model_D, best_acc_D, hist_D = train_model(
    model=model_D,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=10,
    lr=1e-4,   # fine-tuning usually use a smaller lr
    save_path="../models/best_model_D.pth"
)
results["Model D"] = best_acc_D

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(range(1, len(hist_A["train_loss"])+1), hist_A["train_loss"], label="Model A (Linear probing)")
plt.plot(range(1, len(hist_B["train_loss"])+1), hist_B["train_loss"], label="Model B (Fine-tuning)")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Dynamics (Representative Models)")
plt.legend()
plt.tight_layout()
plt.savefig("loss_A_vs_B.png", dpi=200)
plt.show()

In [17]:
!python --version

Python 3.13.5


In [15]:
pip show torch torchvision

Name: torch
Version: 2.6.0+cu124
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: C:\Users\Elahe\Python\miniAnacoda\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: fastai, timm, torchaudio, torchmetrics, torchvision, xai-proj-b
---
Name: torchvision
Version: 0.21.0+cu124
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
Author-email: soumith@pytorch.org
License: BSD
Location: C:\Users\Elahe\Python\miniAnacoda\Lib\site-packages
Requires: numpy, pillow, torch
Required-by: fastai, timm, xai-proj-b
Note: you may need to restart the kernel to use updated packages.


In [18]:
!nvidia-smi

Sat Jan 10 19:38:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 577.00                 Driver Version: 577.00         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 Ti   WDDM  |   00000000:02:00.0 Off |                  N/A |
|  0%   27C    P8              6W /  180W |     356MiB /  16311MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

📝 Notes for Contributors

Each block should be completed in a separate Git branch

Add your name like:
Assigned to: Fatemeh

After finishing a block, create a Pull Request

Do not modify others’ blocks unless discussed as a team

In [28]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))  # مثلا (8, 6) یعنی sm_86


NVIDIA GeForce RTX 5060 Ti
(12, 0)
